In [17]:
!pip install -q qiskit
!pip install -q qiskit-aer
!pip install -q qiskit-algorithms
!pip install -q qiskit-nature
!pip install -q qiskit-nature-pyscf
!pip install -q pyscf
!pip install -q scipy matplotlib

import numpy as np
from scipy.linalg import eigh
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.operators import FermionicOp

print("All imports successful!")

All imports successful!


In [18]:
h = SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IZII', 'ZIII', 'IIZZ', 'IZIZ', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.67810865+0.j,  0.07683157+0.j, -0.07802605+0.j,  0.07683157+0.j,
 -0.07802605+0.j,  0.08418098+0.j,  0.12685108+0.j,  0.12720766+0.j,
  0.04302668+0.j,  0.04302668+0.j,  0.04302668+0.j,  0.04302668+0.j,
  0.12720766+0.j,  0.13086927+0.j,  0.08418098+0.j])

hamiltonian = h
print(f"Hamiltonian: {hamiltonian.num_qubits} qubits, {len(hamiltonian)} terms")

Hamiltonian: 4 qubits, 15 terms


In [19]:
statevector_array = np.array([
    -1.20435792e-03-1.24000232e-03j,
     5.90184494e-03-1.68564974e-03j,
     1.44348600e-03+4.39195673e-04j,
    -5.58947069e-03-4.50609562e-03j,
    -8.49377756e-04+2.29394815e-04j,
     6.97860409e-01+6.70071228e-01j,
    -3.93437104e-03-1.58009705e-03j,
    -6.57431026e-03+3.82047989e-03j,
    -6.43745293e-04-2.70724107e-04j,
     4.72150010e-03+1.25159244e-03j,
    -1.80238812e-01-1.73903623e-01j,
    -1.15342310e-03+5.84675480e-04j,
     6.86917125e-03+3.17232114e-02j,
    -1.10959401e-04+8.21756795e-04j,
    -1.03189882e-03+3.76601683e-03j,
    -5.46607973e-04-2.15358538e-04j
])

statevector = Statevector(statevector_array)

# Verify energy
vqe_energy = statevector.expectation_value(hamiltonian).real
print(f"VQE energy from statevector: {vqe_energy:.8f} Ha")
print(f"Statevector norm: {np.linalg.norm(statevector_array):.6f}")

VQE energy from statevector: -1.19723337 Ha
Statevector norm: 1.000000


In [20]:
# Build excitation pool with identity
n_spin = 4

# Identity operator (critical for ground state inclusion)
op_identity = FermionicOp({'' : 1.0}, num_spin_orbitals=n_spin)

# Single excitations (spin-conserving)
op_single_alpha = FermionicOp({'+_2 -_0': 1.0}, num_spin_orbitals=n_spin)  # α: 0→1
op_single_beta  = FermionicOp({'+_3 -_1': 1.0}, num_spin_orbitals=n_spin)   # β: 0→1

# Double excitation (move both electrons)
op_double = FermionicOp({'+_2 +_3 -_1 -_0': 1.0}, num_spin_orbitals=n_spin)

# Pool with identity first
fermi_pool = [op_identity, op_single_alpha, op_single_beta, op_double]

# Map to Pauli strings
mapper = JordanWignerMapper()
pauli_pool = [mapper.map(op) for op in fermi_pool]

print(f"Number of operators: {len(pauli_pool)}")
for i, op in enumerate(pauli_pool):
    print(f"O_{i}: {op}")

Number of operators: 4
O_0: SparsePauliOp(['IIII'],
              coeffs=[1.+0.j])
O_1: SparsePauliOp(['IXZY', 'IXZX', 'IYZY', 'IYZX'],
              coeffs=[0.  +0.25j, 0.25+0.j  , 0.25+0.j  , 0.  -0.25j])
O_2: SparsePauliOp(['XZYI', 'XZXI', 'YZYI', 'YZXI'],
              coeffs=[0.  +0.25j, 0.25+0.j  , 0.25+0.j  , 0.  -0.25j])
O_3: SparsePauliOp(['XYXY', 'XYXX', 'XYYY', 'XYYX', 'YYXY', 'YYXX', 'YYYY', 'YYYX', 'XXXY', 'XXXX', 'XXYY', 'XXYX', 'YXXY', 'YXXX', 'YXYY', 'YXYX'],
              coeffs=[ 0.0625+0.j    ,  0.    -0.0625j,  0.    +0.0625j,  0.0625+0.j    ,
  0.    -0.0625j, -0.0625+0.j    ,  0.0625+0.j    ,  0.    -0.0625j,
  0.    +0.0625j,  0.0625+0.j    , -0.0625+0.j    ,  0.    +0.0625j,
  0.0625+0.j    ,  0.    -0.0625j,  0.    +0.0625j,  0.0625+0.j    ])


In [21]:
# Build H and S matrices
n_ops = len(pauli_pool)
H_mat = np.zeros((n_ops, n_ops), dtype=complex)
S_mat = np.zeros((n_ops, n_ops), dtype=complex)

for i in range(n_ops):
    Oi = pauli_pool[i]
    Oi_dag = Oi.conjugate().transpose()
    for j in range(n_ops):
        Oj = pauli_pool[j]

        # Overlap matrix: <Ψ| Oi† Oj |Ψ>
        overlap_op = Oi_dag @ Oj
        S_val = statevector.expectation_value(overlap_op)
        S_mat[i, j] = S_val

        # Hamiltonian matrix: <Ψ| Oi† H Oj |Ψ>
        H_Oj = hamiltonian @ Oj
        full_op = Oi_dag @ H_Oj
        H_val = statevector.expectation_value(full_op)
        H_mat[i, j] = H_val

# Take real parts (should be Hermitian)
S_real = np.real(S_mat)
H_real = np.real(H_mat)

print("S matrix (should have 1 on diagonal for identity):")
print(S_real)
print("\nH matrix:")
print(H_real)

S matrix (should have 1 on diagonal for identity):
[[ 1.00000000e+00  3.42345155e-05  4.02040145e-05 -1.81342855e-04]
 [ 3.42345155e-05  1.14751546e-04  2.05537708e-05 -3.20304816e-05]
 [ 4.02040145e-05  2.05537708e-05  1.29617229e-04 -2.91111200e-05]
 [-1.81342855e-04 -3.20304816e-05 -2.91111200e-05  5.15470803e-05]]

H matrix:
[[-1.19723337e+00 -3.72256153e-05 -4.28233749e-05  1.85311065e-04]
 [-3.72256153e-05 -1.00324989e-04 -2.98751347e-05  3.27313842e-05]
 [-4.28233749e-05 -2.98751347e-05 -1.11699253e-04  2.97481400e-05]
 [ 1.85311065e-04  3.27313842e-05  2.97481400e-05 -5.26750521e-05]]


In [22]:
# Verification
print(f"S is Hermitian: {np.allclose(S_real, S_real.T)}")
S_eigvals = np.linalg.eigvalsh(S_real)
print(f"Smallest eigenvalue of S: {S_eigvals.min():.6f} (should be > 0)")

# Check that identity operator gives correct energy
# The first row/column should correspond to identity
# The (0,0) element of H should equal VQE energy
print(f"\nH[0,0] (from identity-identity): {H_real[0,0]:.8f} Ha")
print(f"VQE energy: {vqe_energy:.8f} Ha")
print(f"Difference: {abs(H_real[0,0] - vqe_energy):.2e} Ha")

S is Hermitian: True
Smallest eigenvalue of S: 0.000034 (should be > 0)

H[0,0] (from identity-identity): -1.19723337 Ha
VQE energy: -1.19723337 Ha
Difference: 0.00e+00 Ha


In [23]:
# --- Solve H v = E S v ---
try:
    eigvals, eigvecs = eigh(H_real, S_real)
except np.linalg.LinAlgError as e:
    print("S is singular. Adding small regularization...")
    S_reg = S_real + 1e-10 * np.eye(len(S_real))
    eigvals, eigvecs = eigh(H_real, S_reg)

print("\nEigenvalues (Hartree):")
for i, val in enumerate(eigvals):
    print(f"  E_{i} = {val:.8f} Ha")


Eigenvalues (Hartree):
  E_0 = -1.19736017 Ha
  E_1 = -1.02188236 Ha
  E_2 = -0.92877744 Ha
  E_3 = -0.74874947 Ha


In [24]:
# Extract excitation energy
E0_qse = eigvals[0]
E1_qse = eigvals[1] if len(eigvals) > 1 else None

print(f"QSE ground energy: {E0_qse:.8f} Ha")
print(f"VQE ground energy: {vqe_energy:.8f} Ha")
print(f"Difference: {abs(E0_qse - vqe_energy):.2e} Ha")

if E1_qse is not None:
    delta_E_ha = E1_qse - E0_qse
    delta_E_ev = delta_E_ha * 27.2114
    print(f"\nFirst excited energy: {E1_qse:.8f} Ha")
    print(f"Vertical excitation energy: {delta_E_ev:.4f} eV")
    print(f"Literature value: ~7.6 eV")
    print(f"Deviation: {abs(delta_E_ev - 7.6):.4f} eV")
else:
    print("Only one eigenvalue found – need more operators.")

QSE ground energy: -1.19736017 Ha
VQE ground energy: -1.19723337 Ha
Difference: 1.27e-04 Ha

First excited energy: -1.02188236 Ha
Vertical excitation energy: 4.7750 eV
Literature value: ~7.6 eV
Deviation: 2.8250 eV
